# Artemide — free from-scratch training experiment

Run this notebook on Kaggle's free GPU when available. It trains Artemide's own small decoder-only Transformer from random initialization; it does not download pretrained model weights or call an external AI API.

**Reality check:** Tiny Shakespeare is a small public-domain text corpus. This experiment teaches a tiny model to imitate Shakespeare-style text; it does not create a general-purpose assistant. The checkpoint is a research artifact, not a production model/API.

In Kaggle: set **Accelerator → GPU**, enable Internet, then run all cells. Save the final notebook version to preserve the output artifact. GPU availability and session time are limited by Kaggle.

In [ ]:
import os, sys, json, pathlib, subprocess, textwrap, urllib.request
REPO = pathlib.Path('/kaggle/working/Artemm')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/DevUpreti0/Artemm.git',str(REPO)],check=True)
os.chdir(REPO)
print('Repository:', REPO)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: GPU is not enabled. This small experiment can run on CPU, but will be slower.')

In [ ]:
# Install the repository package without replacing Kaggle's preinstalled PyTorch.
subprocess.run([sys.executable,'-m','pip','install','-e','./core-v2.5[api]','--no-deps'],check=True)
print('Artemide package installed.')

In [ ]:
# Download a small public-domain Shakespeare corpus (raw text only; no pretrained weights).
data_dir = pathlib.Path('/kaggle/working/shakespeare_docs')
data_dir.mkdir(exist_ok=True)
source_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
raw_path = pathlib.Path('/kaggle/working/tinyshakespeare.txt')
urllib.request.urlretrieve(source_url, raw_path)
text = raw_path.read_text(encoding='utf-8')
paragraphs = [p.strip() for p in text.split('\\n\\n') if p.strip()]
# Separate documents permit a document-level validation split rather than random token leakage.
for i, paragraph in enumerate(paragraphs):
    (data_dir / f'doc_{i:05d}.txt').write_text(paragraph, encoding='utf-8')
print('Characters:', len(text), '| documents:', len(paragraphs))

In [ ]:
config = {
    'vocab_size': 259,
    'd_model': 64,
    'n_layers': 2,
    'n_heads': 4,
    'max_seq_len': 256,
    'dropout': 0.0,
    'ff_mult': 4
}
config_path = pathlib.Path('/kaggle/working/artemide-small.json')
config_path.write_text(json.dumps(config, indent=2))
from artemide.data import prepare_corpus
manifest = prepare_corpus(data_dir, '/kaggle/working/artemide-data', validation_fraction=0.05, seed=42, license_name='public-domain Shakespeare text; source attribution recorded in notebook')
print(json.dumps(manifest, indent=2))

In [ ]:
# Train from random initialization. On Kaggle's P100/T4 GPU, start with this modest run.
from artemide.model.config import ModelConfig
from artemide.model.transformer import ArtemideLM
from artemide.trainer import TokenStream, run_training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ArtemideLM(ModelConfig(**config)).to(device)
train_data = TokenStream('/kaggle/working/artemide-data/train.bin', config['vocab_size'])
val_data = TokenStream('/kaggle/working/artemide-data/validation.bin', config['vocab_size'])
result = run_training(
    model, train_data,
    output_dir='/kaggle/working/artemide-run',
    steps=1500, batch_size=32, sequence_length=256,
    learning_rate=3e-4, warmup_steps=100,
    precision='auto', validation_data=val_data,
    validation_interval=150, validation_batches=5,
    checkpoint_interval=500, seed=42,
)
print(json.dumps(result, indent=2))

In [ ]:
# Inspect the validation metrics and save a compact artifact bundle.
metrics_path = pathlib.Path('/kaggle/working/artemide-run/metrics.jsonl')
metrics = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
print('First metrics:', metrics[:2])
print('Last metrics:', metrics[-2:])
assert pathlib.Path('/kaggle/working/artemide-run/latest.pt').is_file()
print('Checkpoint ready:', pathlib.Path('/kaggle/working/artemide-run/latest.pt').stat().st_size, 'bytes')
print('In Kaggle, use the Output panel / Save Version to preserve latest.pt and metrics.jsonl.')

## What happens next?

The checkpoint is trained only on this small corpus and is not a chat assistant. The next engineering step is exporting Artemide's architecture to ONNX and adding browser-side inference with ONNX Runtime Web/WebGPU. That would let each visitor's own device run inference, avoiding a permanently running cloud server. It is not implemented by this notebook yet.